# Imports, Vars and Functions

In [0]:
%pip install lightgbm catboost optuna

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px # one-liner charts, high-level
import plotly.graph_objects as go # full control chart

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.base import clone

from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier, Pool

import optuna

import typing

import warnings

## Vars

In [0]:
TARGET = "Exited"
N_SPLITS = 5

categorical_columns = ["Geography", "Gender"]
numerial_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
]

drop_columns  = ["id", "CustomerId", "source"]
submission_columns = ["id", "Exited"]

## Functions

### Load Data

In [0]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Bank Churn/train.csv')
    test_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Bank Churn/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

### Data splitter

In [0]:
def data_spliter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train = df[df['source'] == 'train'].drop(columns=drop_columns)
    submission_df = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train.drop(columns=[TARGET])
    y_train = train[TARGET]

    return X_train, y_train, submission_df

In [0]:
train_df, test_df, df = load_data(option="colab")

# EDA

In [0]:
len(df)

-   **Customer** ID: Уникальный идентификатор каждого клиента.
-   **Surname**: Фамилия клиента.
-   **Credit Score**: Числовое значение, представляющее кредитный рейтинг клиента.
-   **Geography**: Страна проживания клиента (Франция, Испания или Германия).
-   **Gender**: Пол клиента (Мужской или Женский).
-   **Age**: Возраст клиента.
-   **Tenure**: Количество лет, которое клиент обслуживается в банке.
-   **Balance**: Баланс на счёте клиента.
-   **NumOfProducts**: Количество банковских продуктов, которыми пользуется клиент (например, сберегательный счёт, кредитная карта).
-   **HasCrCard**: Наличие кредитной карты у клиента (1 = да, 0 = нет).
-   **IsActiveMember**: Является ли клиент активным членом банка (1 = да, 0 = нет).
-   **EstimatedSalary**: Предполагаемая заработная плата клиента.
-   **Exited**: Ушёл ли клиент (1 = да, 0 = нет).


In [0]:
df.info()

In [0]:
df.describe().T

In [0]:
df.isna().sum()

In [0]:
df.head()

## Pairplot

In [0]:
fig = sns.pairplot(
    data=train_df[numerial_columns + [TARGET]],
    hue=TARGET,
    diag_kind="kde",
    plot_kws={"alpha": 0.5, "s": 15, "edgecolor": None},
    diag_kws={"fill": True, "alpha": 0.6},
    palette={0: "#2196F3", 1: "#FF5722"},
    # corner=True,
)
fig.figure.suptitle("Pairplot of Numerical Features by Churn Status", y=1.02, fontsize=16, fontweight="bold")

## Correlations

In [0]:
corr_df = pd.get_dummies(
        data=df.drop(columns=drop_columns + ["source"] + ["Surname"]),
        columns=["Gender", "Geography"],
    ).corr()
sorted_columns = corr_df[TARGET].abs().sort_values(ascending=False).index.tolist()
corr_df = corr_df.loc[sorted_columns, sorted_columns]
corr_mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)

plt.figure(figsize=(10,8))
sns.heatmap(
    data=corr_df,
    mask=corr_mask,
    annot=True,
    fmt='.2f',
    cmap="coolwarm",
    square=True,
    center=0
)

In [0]:
corr_df[corr_mask] = np.nan
px.imshow(
    corr_df,
    text_auto= ".2f",
    aspect = True,
    color_continuous_scale="RdBu_r"
).update_traces(hoverongaps=False)

## Surnames

In [0]:
display(df['Surname'].value_counts().reset_index())

In [0]:
df[df['Surname'].str.endswith('ov') | df['Surname'].str.endswith('ova') | df['Surname'].str.endswith('ev') | df['Surname'].str.endswith('eva')]

In [0]:
df[df['Surname'].str.contains("?", regex=False)]

In [0]:
display(df[df['Surname'].str.contains("'", regex=False)])

## Balance

In [0]:
plt.figure(figsize=(16,9))
sns.histplot(
    data=train_df,
    x="Balance",
    hue='Exited',
    bins=50,
    multiple='fill',
)

In [0]:
df['Balance'].drop_duplicates().sort_values(ascending=False).head()

In [0]:
df[df['Balance'] > 200000]

## Salary

In [0]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=train_df,
    x="EstimatedSalary",
    hue='Exited',
    bins=50,
    multiple='fill',
)

## Age

In [0]:
px.histogram(
    data_frame=df,
    x = 'Age',
    # y='Balance',
    color='Exited',
    barmode='overlay',
    opacity=0.6,
    marginal="box",
    # histfunc="avg"
    # nbins=70,
    # text_auto=True
    # range_x=[17,74]
).update_layout(bargap=0.1)

In [0]:
df['Age'].value_counts().reset_index().sort_values(by='Age').head()

In [0]:
plt.figure(figsize=(12,5))
ax = sns.countplot(
    data=train_df,
    x=df['Age'].astype(int),
    hue='Exited',
)

In [0]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=train_df,
    x="Age",
    hue='Exited',
    bins=10,
    # multiple='fill',
)
# plt.xticks(np.arange(18,75))
plt.tight_layout()

## Credit Score

In [0]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=train_df,
    x="CreditScore",
    hue='Exited',
    bins=5,
    multiple='fill',
)
# plt.xticks(np.arange(18,75))
plt.tight_layout()

## Geography

In [0]:
px.histogram(
    df,
    x='Geography',
    color = 'Exited',
    barmode='overlay',
    opacity=0.6,
)

## Num Products

In [0]:
px.histogram(
    df,
    x='NumOfProducts',
    color = 'Exited',
    barmode='overlay',
    opacity=0.6,
)

# Feature Engineering

In [0]:
train_df, test_df, df = load_data(option='colab')

In [0]:
# TODO
# NumOfProducts is not ordinal, add is_3_or_4_products or set as catgory
# Age × IsActiveMember is a sensible interaction: inactive older customers are the highest-risk group.
def feature_engineering(df):
  original_columns = df.columns.tolist()
  print("Engineering features...")

  # 1. Interaction & Ratio Features
  df['Balance_to_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
  df['Age_to_Tenure_Ratio'] = df['Tenure'] / (df['Age'] + 1)
  df['Products_per_Tenure'] = df['NumOfProducts'] / (df['Tenure'] + 1)

  df['Age_Activity_Combo'] = df['Age'] * df['IsActiveMember']

  # 2. Financial Status & Behavior Indicators
  df['Is_Zero_Balance'] = (df['Balance'] == 0).astype(int)


  # 3. Engagement Score
  df['Customer_Activity_Index'] = df['HasCrCard'] + df['IsActiveMember'] + (df['NumOfProducts'] > 1).astype(int)

  # 4. Surname & Demographic Patterns
  # Surname special character rules
  df['Surname_Has_Question'] = df['Surname'].str.contains('?', regex=False).astype(int)

  # Endings often representing specific ethnicities/families in synthetic datasets
  df['Surname_Ends_With_Slavic'] = df['Surname'].str.endswith(('ov', 'ova', 'ev', 'eva')).astype(int)

  df['Surname_Contains_Apostrophe'] = df['Surname'].str.contains("'", regex=False).astype(int)

  # Surname legnth
  df['Surname_Length'] = df['Surname'].str.len()

  # Surname first and last characters
  df['Surname_Start'] = df['Surname'].str[0].str.lower().astype('category')
  df['Surname_End'] = df['Surname'].str[-1].str.lower().astype('category')

  # Convert gended to numbers
  df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

  # Make Geography categorical column:
  df['Geography'] = df['Geography'].astype("category")

  # Num of products 3 or 4
  df['Is_products_3_or_4'] = (df['NumOfProducts'] > 2).astype(int)


  engineered_features = [k for k in df.columns.tolist() if k not in original_columns]

  print(f"\nAdded {len(engineered_features)} features")
  print(f"Features added:")
  for idx, f in enumerate(engineered_features):
    print(f"{idx + 1}: {f}")
  return df

In [0]:
df = feature_engineering(df)

In [0]:
df.info()

In [0]:
with pd.option_context('display.max_columns', None):
    display(df.head())

In [0]:
px.histogram(
    df,
    x='Surname_Length',
    color = 'Exited',
    barmode='overlay',
    opacity=0.6,
)

# Modelling

In [0]:
# For LightGBM
# X, y, submission_df = data_spliter(df, additional_drop_columns=['Surname'])

# For Catboost
X, y, submission_df = data_spliter(df)

In [0]:
with pd.option_context('display.max_columns', None):
  display(X.head(5))

In [0]:
cv = StratifiedKFold(
    n_splits = N_SPLITS,
    shuffle = True,
    random_state=42
)

## LightGBM

In [0]:
# lgb_params = {
#     "n_estimators":10000,
#     "learning_rate": 0.03,
#     "num_leaves": 27,
#     "min_child_samples": 30,
#     "colsample_bytree": 0.6,
#     "subsample": 0.8,
#     "subsample_freq": 1
# }

In [0]:
# preprocess = ColumnTransformer(
#     [("cat", OneHotEncoder(handle_unknown="ignore"), ['Geography'])],
#     remainder="passthrough"
# )

In [0]:
def objective_lgb(trial):
    max_depth = trial.suggest_int("max_depth", 4, 12)
    params = {
        "n_estimators": 3000,          # fixed; early stopping decides the real count
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, min(255, 2 ** max_depth)),
        "max_depth": max_depth,
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    oof_local = np.zeros(len(X))
    iters = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
            X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]


            model = LGBMClassifier(**params)
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                categorical_feature=['Geography','Surname_Start','Surname_End'],
                eval_metric="auc",
                callbacks=[early_stopping(100, verbose=False)],
            )

            oof_local[val_idx] = model.predict_proba(X_val)[:, 1]
            iters.append(model.best_iteration_)

            # let Optuna kill hopeless trials early
            trial.report(roc_auc_score(y_val, oof_local[val_idx]), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()


    trial.set_user_attr("best_iters", iters)
    return roc_auc_score(y, oof_local)

In [0]:
# Re-split X and y to include the newly engineered features
# X, y, submission_df = data_spliter(df)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective_lgb, n_trials=100, show_progress_bar=True)

print(study.best_value)
print(study.best_params)
# how much of the top is noise?
print(study.trials_dataframe().sort_values("value", ascending=False)["value"].head(15).to_string())

In [0]:
oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_df))
fold_train_results, fold_validation_results = [], []
best_iters = []

X_sub_raw = submission_df.drop(columns=["id", "CustomerId"])

In [0]:
# lgb_params_optuna_1 = {'n_estimators': 10000, 'learning_rate': 0.02757359293934948, 'num_leaves': 244, 'max_depth': 10, 'min_child_samples': 128, 'subsample': 0.6624074561769746, 'subsample_freq': 1, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}

# lgb_params_optuna_2 = {'n_estimators': 10000,'learning_rate': 0.014588395647499933, 'num_leaves': 106, 'max_depth': 4, 'min_child_samples': 113, 'subsample': 0.7534735323110857, 'subsample_freq': 4, 'colsample_bytree': 0.5332132855147186, 'reg_alpha': 4.304692668736199e-07, 'reg_lambda': 2.4512199118681435e-05}

# lgb_params_optuna_3 = {'n_estimators': 10000, 'learning_rate': 0.032442034205477026, 'num_leaves': 91, 'max_depth': 4, 'min_child_samples': 61, 'subsample': 0.7251686814955571, 'subsample_freq': 3, 'colsample_bytree': 0.6761636926528118, 'reg_alpha': 0.49537613035821393, 'reg_lambda': 0.06011464654656114}

# lgb_params_optuna_new_feat = {'n_estimators': 3000, 'learning_rate': 0.033950534076790606, 'num_leaves': 206, 'max_depth': 4, 'min_child_samples': 59, 'subsample': 0.6944600326600292, 'subsample_freq': 2, 'colsample_bytree': 0.62670158867226, 'reg_alpha': 3.653660050190367, 'reg_lambda': 0.02918859219237241}

lgb_params_optuna_new_feat_2 = {'n_estimators': 3000, 'max_depth': 4, 'learning_rate': 0.03755714668175416, 'num_leaves': 16, 'min_child_samples': 118, 'subsample': 0.8960672366371799, 'subsample_freq': 5, 'colsample_bytree': 0.5117178113431882, 'reg_alpha': 5.7764356817014716e-08, 'reg_lambda': 1.5042296595462026e-08}

In [0]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
  with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]


    model = LGBMClassifier(**lgb_params_optuna_new_feat_2, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=['Geography','Surname_Start','Surname_End'],
        eval_metric="auc",
        callbacks=[early_stopping(100, verbose=False)],
    )

    oof_pred = model.predict_proba(X_val)[:, 1]
    train_pred = model.predict_proba(X_train)[:, 1]

    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    fold_train_results.append(train_auc)
    fold_validation_results.append(oof_auc)
    best_iters.append(model.best_iteration_)

    submission_results += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS

    print(f"======= FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc:.5f} | VAL AUC: {oof_auc:.5f} | "
          f"DELTA: {oof_auc - train_auc:+.5f} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(fold_train_results), np.array(fold_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iters):.0f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

In [0]:
imp = pd.DataFrame({
    "feature": model.feature_name_,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

imp

In [0]:
import lightgbm as lgb
lgb.plot_importance(model, importance_type="gain", max_num_features=20, figsize=(8, 6))

## Catboost

In [0]:
# For Catboost
X, y, submission_df = data_spliter(df)

In [0]:
with pd.option_context('display.max_columns', None):
  display(X.head(5))

In [0]:
catboost_categories = ['Surname', 'Geography', 'NumOfProducts', 'Surname_Length', 'Surname_Start', 'Surname_End']

X[catboost_categories] = X[catboost_categories].astype('str')
submission_df[catboost_categories] = submission_df[catboost_categories].astype('str')

### Optuna

In [0]:
def objective_catboost(trial):

    params = {
        'iterations': 5000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth': trial.suggest_int('depth', 3, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.5, 30.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.0, 5.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 30, 150),
        'eval_metric': 'AUC',
        'early_stopping_rounds': 200,
        'use_best_model': True,
        'allow_writing_files': False,
        'random_state': 42,
        'verbose': 0,
    }

    bt = trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli'])
    params['bootstrap_type'] = bt
    if bt == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 2.0)
    else:
        params['subsample'] = trial.suggest_float('subsample', 0.6, 1.0)

    oof_local = np.zeros(len(X))
    iters = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
            y_train = y.iloc[tr_idx]
            y_val =  y.iloc[val_idx]

            train_pool = Pool(X.iloc[tr_idx], y_train, cat_features=catboost_categories)
            valid_pool = Pool(X.iloc[val_idx], y_val, cat_features=catboost_categories)


            model = CatBoostClassifier(**params)
            model.fit(
                train_pool,
                eval_set=valid_pool
            )

            oof_local[val_idx] = model.predict_proba(valid_pool)[:, 1]
            iters.append(model.best_iteration_)

            # let Optuna kill hopeless trials early
            trial.report(roc_auc_score(y_val, oof_local[val_idx]), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()


    trial.set_user_attr("best_iters", iters)
    return roc_auc_score(y, oof_local)

In [0]:
catboost_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
catboost_study.optimize(objective_catboost, n_trials=100, show_progress_bar=True)

print(catboost_study.best_value)
print(catboost_study.best_params)
# how much of the top is noise?
print(catboost_study.trials_dataframe().sort_values("value", ascending=False)["value"].head(15).to_string())

In [0]:
print(catboost_study.best_value)
print(catboost_study.best_params)

### Final Prediction

In [0]:
oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_df))
fold_train_results, fold_validation_results = [], []
best_iters = []

X_sub_raw = submission_df.drop(columns=["id", "CustomerId"])
sub_pool = Pool(X_sub_raw[X.columns], cat_features=catboost_categories)

In [0]:
catboost_optuna_1 = {'iterations':5000, 'learning_rate': 0.012707942999213693, 'depth': 4, 'l2_leaf_reg': 0.6017151754328816, 'random_strength': 1.6266516538163218, 'min_data_in_leaf': 77, 'bootstrap_type': 'Bernoulli', 'subsample': 0.7427013306774357,         'use_best_model': True,
        'allow_writing_files': False,
        'random_state': 42,
        'verbose': 0,}

In [0]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
  with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    y_train = y.iloc[tr_idx]
    y_val =  y.iloc[val_idx]

    train_pool = Pool(X.iloc[tr_idx], y_train, cat_features=catboost_categories)
    valid_pool = Pool(X.iloc[val_idx], y_val, cat_features=catboost_categories)


    model = CatBoostClassifier(
        **catboost_optuna_1,
        # iterations=5000,
        # learning_rate = 0.037,
        # depth = 4,
        # l2_leaf_reg=1.5,
        # subsample=0.85,
        eval_metric='AUC',
        early_stopping_rounds=200,
        # verbose=0,
        # allow_writing_files=False,
        # random_state=42,
        # use_best_model=True
    )
    model.fit(
        train_pool,
        eval_set=valid_pool,
        verbose=100
    )

    oof_pred = model.predict_proba(valid_pool)[:, 1]
    train_pred = model.predict_proba(train_pool)[:, 1]

    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    fold_train_results.append(train_auc)
    fold_validation_results.append(oof_auc)
    best_iters.append(model.best_iteration_)

    submission_results += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS

    print(f"======= FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc:.5f} | VAL AUC: {oof_auc:.5f} | "
          f"DELTA: {oof_auc - train_auc:+.5f} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(fold_train_results), np.array(fold_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iters):.0f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

# Submissoin

In [0]:
submission_df[TARGET] = submission_results

In [0]:
lgb_prediction = submission_df[['id', TARGET]]

In [0]:
catboost_predictons = submission_df[['id', TARGET]]

In [0]:
# lgb_params = {
#     "n_estimators":10000,
#     "learning_rate": 0.03,
#     "num_leaves": 24,
#     "min_child_samples": 30,
#     "colsample_bytree": 0.6,
#     "subsample": 0.7,
#     "subsample_freq": 1
# }

In [0]:
# lgb_prediction.to_csv('lgb_prediction_v6.csv', index=False)

In [0]:
catboost_predictons.to_csv('catboost_prediction_v3.csv', index=False)